# Notebook 02 — Funnel Validation

## 1. Setup

In [1]:
from pathlib import Path
import duckdb
import pandas as pd
import numpy as np
from IPython.display import display

PROJECT_ROOT = Path("/Users/abrahameshete/dsp-campaign-analyst")
# If this notebook is opened from a different directory, set PROJECT_ROOT manually:
# PROJECT_ROOT = Path("/Users/your-user/dsp-campaign-analyst")

DB_PATH = PROJECT_ROOT / "data" / "processed" / "campaign_analysis.duckdb"
STAGING_DIR = PROJECT_ROOT / "pipeline" / "sql" / "staging"
ADVERTISER_ID = "1458"

if not DB_PATH.exists():
    raise FileNotFoundError(
        f"DuckDB database not found: {DB_PATH}\n"
        "Open this notebook from the project root, or update PROJECT_ROOT."
    )

if not STAGING_DIR.exists():
    raise FileNotFoundError(f"Staging SQL folder not found: {STAGING_DIR}")

con = duckdb.connect(str(DB_PATH), read_only=False)
print(f"Connected to: {DB_PATH}")


Connected to: /Users/abrahameshete/dsp-campaign-analyst/data/processed/campaign_analysis.duckdb


## 2. Build the staging tables

In [7]:
# Rebuild temporary staging tables from the existing SQL.
# This keeps the notebook aligned with the pipeline's transformations.
for sql_file in sorted(STAGING_DIR.glob("stg_*.sql")):
    table_name = sql_file.stem
    sql = sql_file.read_text()
    con.execute(f"CREATE OR REPLACE TEMP TABLE {table_name} AS {sql}")
    count = con.execute(f"SELECT COUNT(*) FROM {table_name}").fetchone()[0]
    print(f"{table_name}: {count:,} rows")

required_tables = ["stg_impressions", "stg_clicks", "stg_conversions"]
available_tables = {
    row[0] for row in con.execute("SHOW TABLES").fetchall()
}
missing = [t for t in required_tables if t not in available_tables]
if missing:
    raise RuntimeError(f"Missing required staging tables: {missing}")


stg_bids: 3,250,536 rows
stg_clicks: 347 rows
stg_conversions: 1 rows
stg_impressions: 448,164 rows


## 3. Confirm table grain and inspect columns

In [8]:
def describe_table(table_name):
    return con.execute(f"DESCRIBE {table_name}").df()

for table in ["stg_impressions", "stg_clicks", "stg_conversions"]:
    print(f"\\n{table} — one row should represent one logged {table.replace('stg_', '').rstrip('s')} event")
    display(describe_table(table))


\nstg_impressions — one row should represent one logged impression event


,column_name,column_type,null,key,default,extra
0,bid_id,VARCHAR,YES,None,None,None
1,impression_datetime,TIMESTAMP,YES,None,None,None
2,impression_date,DATE,YES,None,None,None
3,hour,BIGINT,YES,None,None,None
4,ipinyou_id,VARCHAR,YES,None,None,None
5,ad_exchange,BIGINT,YES,None,None,None
6,region,BIGINT,YES,None,None,None
7,city,BIGINT,YES,None,None,None
8,creative_id,VARCHAR,YES,None,None,None
9,ad_slot_width,BIGINT,YES,None,None,None


\nstg_clicks — one row should represent one logged click event


,column_name,column_type,null,key,default,extra
0,bid_id,VARCHAR,YES,None,None,None
1,click_datetime,TIMESTAMP,YES,None,None,None
2,click_date,DATE,YES,None,None,None
3,hour,INTEGER,YES,None,None,None
4,ipinyou_id,VARCHAR,YES,None,None,None
5,ad_exchange,VARCHAR,YES,None,None,None
6,region,VARCHAR,YES,None,None,None
7,city,VARCHAR,YES,None,None,None
8,creative_id,VARCHAR,YES,None,None,None
9,ad_slot_width,INTEGER,YES,None,None,None


\nstg_conversions — one row should represent one logged conversion event


,column_name,column_type,null,key,default,extra
0,bid_id,VARCHAR,YES,None,None,None
1,conversion_datetime,TIMESTAMP,YES,None,None,None
2,conversion_date,DATE,YES,None,None,None
3,hour,INTEGER,YES,None,None,None
4,ipinyou_id,VARCHAR,YES,None,None,None
5,ad_exchange,VARCHAR,YES,None,None,None
6,region,VARCHAR,YES,None,None,None
7,city,VARCHAR,YES,None,None,None
8,creative_id,VARCHAR,YES,None,None,None
9,ad_slot_width,INTEGER,YES,None,None,None


## 4. Reconcile event counts

In [9]:
counts = []
for table in ["stg_impressions", "stg_clicks", "stg_conversions"]:
    n = con.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    counts.append({"event": {"stg_impressions": "impressions", "stg_clicks": "clicks", "stg_conversions": "conversions"}[table], "rows": n})

event_counts = pd.DataFrame(counts)
display(event_counts)

# Known campaign totals from the completed pipeline, used as a reconciliation target.
expected = {"impressions": 448_164, "clicks": 347, "conversions": 1}
observed = {
    row["event"]: int(row["rows"])
    for row in event_counts.to_dict("records")
}
reconciliation = pd.DataFrame([
    {
        "event": event,
        "expected_pipeline_count": count,
        "notebook_count": observed.get(event),
        "matches": observed.get(event) == count,
    }
    for event, count in expected.items()
])
display(reconciliation)


,event,rows
0,impressions,448164
1,clicks,347
2,conversions,1


,event,expected_pipeline_count,notebook_count,matches
0,impressions,448164,448164,True
1,clicks,347,347,True
2,conversions,1,1,True


## 5. Calculate funnel rates

In [10]:
impressions = int(con.execute("SELECT COUNT(*) FROM stg_impressions").fetchone()[0])
clicks = int(con.execute("SELECT COUNT(*) FROM stg_clicks").fetchone()[0])
conversions = int(con.execute("SELECT COUNT(*) FROM stg_conversions").fetchone()[0])

def safe_rate(numerator, denominator):
    return (numerator / denominator * 100) if denominator else np.nan

funnel = pd.DataFrame([
    {"stage": "Impressions", "events": impressions, "rate_from_previous_stage_pct": np.nan},
    {"stage": "Clicks", "events": clicks, "rate_from_previous_stage_pct": safe_rate(clicks, impressions)},
    {"stage": "Conversions", "events": conversions, "rate_from_previous_stage_pct": safe_rate(conversions, clicks)},
])
display(funnel.style.format({
    "events": "{:,.0f}",
    "rate_from_previous_stage_pct": "{:.4f}%"
}))

ctr = safe_rate(clicks, impressions)
click_to_conversion = safe_rate(conversions, clicks)
impression_to_conversion = safe_rate(conversions, impressions)

print(f"CTR (clicks / impressions): {ctr:.4f}%")
print(f"Click-to-conversion rate (conversions / clicks): {click_to_conversion:.4f}%")
print(f"Impression-to-conversion rate: {impression_to_conversion:.6f}%")

print("\\nInterpretation: these are aggregate event-count ratios. They are not proof that every conversion followed a click unless event-level matching supports that claim.")


,stage,events,rate_from_previous_stage_pct
0,Impressions,"448,164",nan%
1,Clicks,347,0.0774%
2,Conversions,1,0.2882%


CTR (clicks / impressions): 0.0774%
Click-to-conversion rate (conversions / clicks): 0.2882%
Impression-to-conversion rate: 0.000223%
\nInterpretation: these are aggregate event-count ratios. They are not proof that every conversion followed a click unless event-level matching supports that claim.


## 6. Find event identifiers for matching

In [11]:
# Identify likely auction/event ID columns without assuming the exact spelling.
ID_CANDIDATES = [
    "bid_id", "bidid", "bidid_id", "bid_id_hash",
    "auction_id", "impression_id", "click_id", "conversion_id"
]

def normalized(name):
    return "".join(ch.lower() for ch in name if ch.isalnum())

def find_id_column(table_name):
    columns = [row[0] for row in con.execute(f"DESCRIBE {table_name}").fetchall()]
    normalized_candidates = {normalized(x) for x in ID_CANDIDATES}
    for col in columns:
        if normalized(col) in normalized_candidates:
            return col
    return None

id_columns = {
    table: find_id_column(table)
    for table in ["stg_impressions", "stg_clicks", "stg_conversions"]
}
display(pd.DataFrame([
    {"table": table, "matching_id_column": col}
    for table, col in id_columns.items()
]))

if not all(id_columns.values()):
    print(
        "At least one staging table has no recognized event ID column. "
        "The notebook will not invent a join key. Review the column descriptions above "
        "and add the actual shared identifier to ID_CANDIDATES if appropriate."
    )


,table,matching_id_column
0,stg_impressions,bid_id
1,stg_clicks,bid_id
2,stg_conversions,bid_id


## 7. Validate click-to-impression matching

In [12]:
click_impression_result = None

click_id = id_columns["stg_clicks"]
impression_id = id_columns["stg_impressions"]

if click_id and impression_id:
    # Count distinct IDs as well as rows. Duplicate IDs are reported separately below.
    click_impression_result = con.execute(f'''
        WITH impression_ids AS (
            SELECT DISTINCT CAST("{impression_id}" AS VARCHAR) AS event_id
            FROM stg_impressions
            WHERE "{impression_id}" IS NOT NULL
        ),
        click_ids AS (
            SELECT CAST("{click_id}" AS VARCHAR) AS event_id
            FROM stg_clicks
            WHERE "{click_id}" IS NOT NULL
        )
        SELECT
            COUNT(*) AS clicks_with_nonnull_id,
            COUNT_IF(i.event_id IS NOT NULL) AS clicks_matching_impression,
            COUNT_IF(i.event_id IS NULL) AS clicks_without_matching_impression
        FROM click_ids c
        LEFT JOIN impression_ids i USING (event_id)
    ''').df()
    display(click_impression_result)
else:
    print("Skipped click-to-impression join: a shared event ID was not identified.")


,clicks_with_nonnull_id,clicks_matching_impression,clicks_without_matching_impression
0,347,345.0,2.0


## 8. Validate conversion-to-click and conversion-to-impression matching

In [13]:
conversion_click_result = None
conversion_impression_result = None

conversion_id = id_columns["stg_conversions"]

if conversion_id and click_id:
    conversion_click_result = con.execute(f'''
        WITH click_ids AS (
            SELECT DISTINCT CAST("{click_id}" AS VARCHAR) AS event_id
            FROM stg_clicks
            WHERE "{click_id}" IS NOT NULL
        ),
        conversion_ids AS (
            SELECT CAST("{conversion_id}" AS VARCHAR) AS event_id
            FROM stg_conversions
            WHERE "{conversion_id}" IS NOT NULL
        )
        SELECT
            COUNT(*) AS conversions_with_nonnull_id,
            COUNT_IF(c.event_id IS NOT NULL) AS conversions_matching_click,
            COUNT_IF(c.event_id IS NULL) AS conversions_without_matching_click
        FROM conversion_ids v
        LEFT JOIN click_ids c USING (event_id)
    ''').df()
    display(conversion_click_result)
else:
    print("Skipped conversion-to-click join: a shared event ID was not identified.")

if conversion_id and impression_id:
    conversion_impression_result = con.execute(f'''
        WITH impression_ids AS (
            SELECT DISTINCT CAST("{impression_id}" AS VARCHAR) AS event_id
            FROM stg_impressions
            WHERE "{impression_id}" IS NOT NULL
        ),
        conversion_ids AS (
            SELECT CAST("{conversion_id}" AS VARCHAR) AS event_id
            FROM stg_conversions
            WHERE "{conversion_id}" IS NOT NULL
        )
        SELECT
            COUNT(*) AS conversions_with_nonnull_id,
            COUNT_IF(i.event_id IS NOT NULL) AS conversions_matching_impression,
            COUNT_IF(i.event_id IS NULL) AS conversions_without_matching_impression
        FROM conversion_ids v
        LEFT JOIN impression_ids i USING (event_id)
    ''').df()
    display(conversion_impression_result)
else:
    print("Skipped conversion-to-impression join: a shared event ID was not identified.")


,conversions_with_nonnull_id,conversions_matching_click,conversions_without_matching_click
0,1,0.0,1.0


,conversions_with_nonnull_id,conversions_matching_impression,conversions_without_matching_impression
0,1,0.0,1.0


## 9. Duplicate event ID checks

In [14]:
duplicate_checks = []

for table in ["stg_impressions", "stg_clicks", "stg_conversions"]:
    id_col = id_columns[table]
    if id_col:
        result = con.execute(f'''
            SELECT
                COUNT(*) AS total_rows,
                COUNT("{id_col}") AS rows_with_id,
                COUNT(DISTINCT "{id_col}") AS distinct_ids,
                COUNT("{id_col}") - COUNT(DISTINCT "{id_col}") AS repeated_id_rows
            FROM {table}
        ''').fetchone()
        duplicate_checks.append({
            "table": table,
            "total_rows": result[0],
            "rows_with_id": result[1],
            "distinct_ids": result[2],
            "repeated_id_rows": result[3],
        })
    else:
        duplicate_checks.append({
            "table": table,
            "total_rows": con.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0],
            "rows_with_id": None,
            "distinct_ids": None,
            "repeated_id_rows": None,
        })

duplicate_report = pd.DataFrame(duplicate_checks)
display(duplicate_report)


,table,total_rows,rows_with_id,distinct_ids,repeated_id_rows
0,stg_impressions,448164,448164,447312,852
1,stg_clicks,347,347,327,20
2,stg_conversions,1,1,1,0


## 10. Timestamp and price sanity checks

In [15]:
# Inspect available timestamp and price columns first; avoid assuming a column exists.
for table in ["stg_impressions", "stg_clicks", "stg_conversions"]:
    cols = [row[0] for row in con.execute(f"DESCRIBE {table}").fetchall()]
    time_cols = [c for c in cols if "time" in c.lower() or "date" in c.lower()]
    price_cols = [c for c in cols if "price" in c.lower() or "cost" in c.lower()]
    print(f"{table}: timestamp/date columns={time_cols}; price/cost columns={price_cols}")


stg_impressions: timestamp/date columns=['impression_datetime', 'impression_date']; price/cost columns=['ad_slot_floor_price', 'bidding_price', 'paying_price']
stg_clicks: timestamp/date columns=['click_datetime', 'click_date']; price/cost columns=['ad_slot_floor_price', 'bidding_price', 'paying_price']
stg_conversions: timestamp/date columns=['conversion_datetime', 'conversion_date']; price/cost columns=['ad_slot_floor_price', 'bidding_price', 'paying_price']


## 11. Daily funnel view

In [16]:
# Discover the date column used by each staging model.
def find_date_column(table_name):
    cols = [row[0] for row in con.execute(f"DESCRIBE {table_name}").fetchall()]
    preferred = [c for c in cols if c.lower().endswith("_date")]
    return preferred[0] if preferred else None

date_columns = {
    "stg_impressions": find_date_column("stg_impressions"),
    "stg_clicks": find_date_column("stg_clicks"),
    "stg_conversions": find_date_column("stg_conversions"),
}
display(pd.DataFrame([{"table": t, "date_column": c} for t, c in date_columns.items()]))

if all(date_columns.values()):
    daily_parts = []
    for table, date_col in date_columns.items():
        event = {
            "stg_impressions": "impressions",
            "stg_clicks": "clicks",
            "stg_conversions": "conversions",
        }[table]
        daily_parts.append(
            f'SELECT CAST("{date_col}" AS DATE) AS event_date, COUNT(*) AS {event} '
            f'FROM {table} GROUP BY 1'
        )

    daily_funnel = con.execute(f"""
        WITH
        daily_impressions AS ({daily_parts[0]}),
        daily_clicks AS ({daily_parts[1]}),
        daily_conversions AS ({daily_parts[2]}),
        dates AS (
            SELECT event_date FROM daily_impressions
            UNION SELECT event_date FROM daily_clicks
            UNION SELECT event_date FROM daily_conversions
        )
        SELECT
            d.event_date,
            COALESCE(i.impressions, 0) AS impressions,
            COALESCE(c.clicks, 0) AS clicks,
            COALESCE(v.conversions, 0) AS conversions
        FROM dates d
        LEFT JOIN daily_impressions i USING (event_date)
        LEFT JOIN daily_clicks c USING (event_date)
        LEFT JOIN daily_conversions v USING (event_date)
        ORDER BY d.event_date
    """).df()
    display(daily_funnel)
else:
    daily_funnel = pd.DataFrame()
    print("Could not identify a date column in every staging table. Review the schema in Section 3.")


,table,date_column
0,stg_impressions,impression_date
1,stg_clicks,click_date
2,stg_conversions,conversion_date


,event_date,impressions,clicks,conversions
0,2013-06-06,448164,347,0
1,2013-06-08,0,0,1


## 12. Findings and interpretation

In [17]:
findings = []

if impressions == 448_164 and clicks == 347 and conversions == 1:
    findings.append("Event counts reconcile with the campaign totals previously loaded to Neon.")
else:
    findings.append("Event counts differ from the previously recorded pipeline totals; investigate the source or staging transformations.")

findings.append(f"Observed CTR is {ctr:.4f}% based on {clicks:,} clicks and {impressions:,} impressions.")
findings.append(f"Observed click-to-conversion ratio is {click_to_conversion:.4f}%; only {conversions:,} conversion event(s) were recorded, so conclusions are highly limited.")

if click_impression_result is not None:
    unmatched = int(click_impression_result.loc[0, "clicks_without_matching_impression"])
    findings.append(f"Clicks without a matching impression ID: {unmatched:,}. These should be flagged and investigated, not silently discarded.")

if conversion_click_result is not None:
    unmatched_conv = int(conversion_click_result.loc[0, "conversions_without_matching_click"])
    findings.append(f"Conversions without a matching click ID: {unmatched_conv:,}.")

if duplicate_report["repeated_id_rows"].notna().any():
    findings.append("Review duplicate-ID counts by event table. A repeated identifier may represent duplicate logging or multiple rows per key; confirm the source grain before removing records.")

for i, finding in enumerate(findings, 1):
    print(f"{i}. {finding}")

print("\\nLimitations:")
print("- Aggregate funnel ratios do not establish event-level attribution.")
print("- A low conversion count makes conversion-rate estimates unstable.")
print("- A join on a shared auction/bid ID is a technical match, not necessarily proof of a unique human user.")
print("- Do not remove unmatched or duplicate rows until the expected source-log grain and business rules are confirmed.")


1. Event counts reconcile with the campaign totals previously loaded to Neon.
2. Observed CTR is 0.0774% based on 347 clicks and 448,164 impressions.
3. Observed click-to-conversion ratio is 0.2882%; only 1 conversion event(s) were recorded, so conclusions are highly limited.
4. Clicks without a matching impression ID: 2. These should be flagged and investigated, not silently discarded.
5. Conversions without a matching click ID: 1.
6. Review duplicate-ID counts by event table. A repeated identifier may represent duplicate logging or multiple rows per key; confirm the source grain before removing records.
\nLimitations:
- Aggregate funnel ratios do not establish event-level attribution.
- A low conversion count makes conversion-rate estimates unstable.
- A join on a shared auction/bid ID is a technical match, not necessarily proof of a unique human user.
- Do not remove unmatched or duplicate rows until the expected source-log grain and business rules are confirmed.


## 13. Export validation outputs

In [18]:
OUTPUT_DIR = PROJECT_ROOT / "reports"
OUTPUT_DIR.mkdir(exist_ok=True)

event_counts.to_csv(OUTPUT_DIR / "notebook02_event_counts.csv", index=False)
reconciliation.to_csv(OUTPUT_DIR / "notebook02_reconciliation.csv", index=False)
funnel.to_csv(OUTPUT_DIR / "notebook02_funnel_metrics.csv", index=False)
duplicate_report.to_csv(OUTPUT_DIR / "notebook02_duplicate_ids.csv", index=False)
daily_funnel.to_csv(OUTPUT_DIR / "notebook02_daily_funnel.csv", index=False)

print(f"Saved validation CSV files to: {OUTPUT_DIR}")


Saved validation CSV files to: /Users/abrahameshete/dsp-campaign-analyst/reports


## Final conclusion

This notebook reconciles the event totals, calculates aggregate funnel rates, tests event-ID relationships, checks duplicate identifiers, and saves reproducible validation outputs.

**Do not describe a conversion as click-attributed unless the event-level ID match confirms it.** Document unmatched records and the low conversion volume as limitations. Raw source data remains unchanged.

Close the DuckDB connection when finished:


In [19]:
con.close()
print("DuckDB connection closed.")


DuckDB connection closed.
